# BirdCLEF 2026 — Self-Supervised Pretraining + Fine-Tuning — Inference Only (Pipeline 04)

This notebook performs **inference only** using a fine-tuned model checkpoint.

In [ ]:
import os, gc, math, glob, numpy as np, pandas as pd, soundfile as sf
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio.transforms as T
import timm
import warnings; warnings.filterwarnings('ignore')

class Config:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')
    MODEL_PATH = 'best_model_ssl_ft.pth'
    SR = 32000
    WINDOW_SECONDS = 5
    N_MELS, N_FFT, HOP_LENGTH, FMIN, FMAX = 128, 2048, 512, 20, 16000
    MODEL_NAME = 'tf_efficientnet_b0'

CFG = Config()

sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
CFG.NUM_CLASSES = len(submission_labels)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
class FineTuneModel(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.encoder = timm.create_model(model_name, pretrained=False, in_chans=3)
        if 'efficientnet' in model_name:
            in_features = self.encoder.classifier.in_features
            self.encoder.classifier = nn.Identity()
        else:
            in_features = self.encoder.get_classifier().in_features
            self.encoder.reset_classifier(0)
        self.head = nn.Linear(in_features, num_classes)
    def forward(self, x): return self.head(self.encoder(x))

model = FineTuneModel(CFG.MODEL_NAME, CFG.NUM_CLASSES).to(device)
try:
    model.load_state_dict(torch.load(CFG.MODEL_PATH, map_location=device))
    model.eval()
    print('Checkpoint loaded successfully.')
except: print('Fallback: using random weights.')


In [ ]:
TEST_DIR = os.path.join(CFG.ROOT_DIR, 'test_soundscapes')
test_files = sorted(glob.glob(f'{TEST_DIR}/*.ogg')) if os.path.exists(TEST_DIR) else []
if not test_files:
    print('FALLBACK: Using train soundscapes')
    test_files = sorted(glob.glob(f'{CFG.SOUNDSCAPE_DIR}/*.ogg'))[:5]

mel_transform = T.MelSpectrogram(sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH, n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX).to(device)
amplitude_to_db = T.AmplitudeToDB(top_db=80).to(device)

all_preds, all_row_ids = [], []

for audio_path in tqdm(test_files):
    filename = os.path.basename(audio_path).replace('.ogg', '')
    try: y, _ = sf.read(audio_path, always_2d=True); y = y.mean(axis=1)
    except: continue
    y_t = torch.tensor(y, dtype=torch.float32).to(device)
    window_samples = CFG.SR * CFG.WINDOW_SECONDS
    for seg_idx in range(math.ceil(len(y_t) / window_samples)):
        start_sample = seg_idx * window_samples
        segment = y_t[start_sample : start_sample + window_samples]
        if len(segment) < window_samples: segment = F.pad(segment, (0, window_samples - len(segment)))
        with torch.no_grad():
            mel = amplitude_to_db(mel_transform(segment))
            mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
            img = torch.stack([mel, mel, mel]).unsqueeze(0)
            probs = torch.sigmoid(model(img)).squeeze(0).cpu().numpy()
        all_row_ids.append(f'{filename}_{(seg_idx + 1) * CFG.WINDOW_SECONDS}')
        all_preds.append(probs)

sub_df = pd.DataFrame(all_preds, columns=submission_labels)
sub_df.insert(0, 'row_id', all_row_ids)
sub_df.to_csv('submission.csv', index=False)
print('Submission saved!')
